<a href="https://colab.research.google.com/github/mohantyk/txt2img/blob/main/Chap4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Download

In [64]:
import os
import zipfile
import requests

# Define the target directory in Colab
base_dir = '/content/files'
os.makedirs(base_dir, exist_ok=True)
print(f"Created directory: {base_dir}")

# --- Andrej Karpathy's dataset ---
karpathy_url = "https://mng.bz/Qw5R"
karpathy_zip_path = os.path.join(base_dir, "dataset_flickr8k.zip")
karpathy_json_target_path = os.path.join(base_dir, "dataset_flickr8k.json")

print(f"\nDownloading Andrej Karpathy's dataset from {karpathy_url}...")
try:
    response = requests.get(karpathy_url, stream=True)
    response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
    with open(karpathy_zip_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"Downloaded {karpathy_zip_path}")

    print(f"Extracting {karpathy_zip_path}...")
    with zipfile.ZipFile(karpathy_zip_path, 'r') as zip_ref:
        # Extract only the dataset_flickr8k.json file
        for file in zip_ref.namelist():
            if file == 'dataset_flickr8k.json':
                zip_ref.extract(file, base_dir)
                print(f"Extracted {file} to {base_dir}")
                break
    os.remove(karpathy_zip_path) # Clean up the zip file
    print(f"Cleaned up {karpathy_zip_path}")

except requests.exceptions.RequestException as e:
    print(f"Error downloading Karpathy dataset: {e}")
except zipfile.BadZipFile:
    print(f"Error: Downloaded file '{karpathy_zip_path}' is not a valid zip file.")
except Exception as e:
    print(f"An unexpected error occurred during Karpathy dataset processing: {e}")


Created directory: /content/files

Downloaded /content/files/dataset_flickr8k.zip
Extracting /content/files/dataset_flickr8k.zip...
Extracted dataset_flickr8k.json to /content/files
Cleaned up /content/files/dataset_flickr8k.zip


In [65]:
# Access Kaggle API key and username from Colab secrets
from google.colab import userdata
import json

KAGGLE_USERNAME = userdata.get('KAGGLE_USERNAME')
KAGGLE_API_KEY = userdata.get('KAGGLE_API') # Assuming 'KAGGLE_API_KEY' is your API key secret

# Construct the kaggle.json content
kaggle_json_content = json.dumps({"username": KAGGLE_USERNAME, "key": KAGGLE_API_KEY})

# Set up Kaggle API credentials
import os

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)

kaggle_json_path = os.path.join(kaggle_dir, 'kaggle.json')
with open(kaggle_json_path, 'w') as f:
    f.write(kaggle_json_content)

os.chmod(kaggle_json_path, 0o600)

print("Kaggle API credentials configured from Colab secrets.")
print("Please ensure you have a 'KAGGLE_USERNAME' secret set to your Kaggle username, ")
print("and a 'KAGGLE_API_KEY' secret set to your raw Kaggle API key string.")
print(f"Content written to {kaggle_json_path}: {kaggle_json_content}")

Kaggle API credentials configured from Colab secrets.
Please ensure you have a 'KAGGLE_USERNAME' secret set to your Kaggle username, 
and a 'KAGGLE_API_KEY' secret set to your raw Kaggle API key string.
Content written to /root/.kaggle/kaggle.json: {"username": "mohantyk", "key": "KGAT_82063f5f018ac2ed987c0e6add14da94"}


In [66]:
# Install Kaggle library
!pip install kaggle -q

print("Kaggle library installed.")

Kaggle library installed.


In [67]:
import zipfile
import os

# Define the target directory (same as before)
base_dir = '/content/files'

# Download the Flickr 8k dataset using Kaggle API
print(f"\nDownloading Flickr 8k dataset to {base_dir} using Kaggle API...")
!kaggle datasets download -d adityajn105/flickr8k -p {base_dir}

# The downloaded file will likely be named flickr8k.zip inside base_dir
flickr8k_zip_path = os.path.join(base_dir, 'flickr8k.zip')

# Create a subdirectory for the extracted images and captions
flickr8k_target_dir = os.path.join(base_dir, 'flickr8k_dataset')
os.makedirs(flickr8k_target_dir, exist_ok=True)

print(f"\nExtracting {flickr8k_zip_path} to {flickr8k_target_dir}...")
with zipfile.ZipFile(flickr8k_zip_path, 'r') as zip_ref:
    zip_ref.extractall(flickr8k_target_dir)
print("Extraction complete.")

# Clean up the zip file after extraction
os.remove(flickr8k_zip_path)
print(f"Cleaned up {flickr8k_zip_path}")

print(f"\nContents of '{base_dir}': {os.listdir(base_dir)}")
print(f"Contents of '{flickr8k_target_dir}': {os.listdir(flickr8k_target_dir)}")


Dataset URL: https://www.kaggle.com/datasets/adityajn105/flickr8k
License(s): CC0-1.0
100% 1.04G/1.04G [00:26<00:00, 42.1MB/s]


Extracting /content/files/flickr8k.zip to /content/files/flickr8k_dataset...
Extraction complete.
Cleaned up /content/files/flickr8k.zip

Contents of '/content/files': ['flickr8k_dataset', 'dataset_flickr8k.json']
Contents of '/content/files/flickr8k_dataset': ['captions.txt', 'Images']


# Data Processing

In [68]:
import json
with open('files/dataset_flickr8k.json', 'r') as fb:
  data = json.load(fb)

In [69]:
for img in data['images']:
  print(img)
  break

{'sentids': [0, 1, 2, 3, 4], 'imgid': 0, 'sentences': [{'tokens': ['a', 'black', 'dog', 'is', 'running', 'after', 'a', 'white', 'dog', 'in', 'the', 'snow'], 'raw': 'A black dog is running after a white dog in the snow .', 'imgid': 0, 'sentid': 0}, {'tokens': ['black', 'dog', 'chasing', 'brown', 'dog', 'through', 'snow'], 'raw': 'Black dog chasing brown dog through snow', 'imgid': 0, 'sentid': 1}, {'tokens': ['two', 'dogs', 'chase', 'each', 'other', 'across', 'the', 'snowy', 'ground'], 'raw': 'Two dogs chase each other across the snowy ground .', 'imgid': 0, 'sentid': 2}, {'tokens': ['two', 'dogs', 'play', 'together', 'in', 'the', 'snow'], 'raw': 'Two dogs play together in the snow .', 'imgid': 0, 'sentid': 3}, {'tokens': ['two', 'dogs', 'running', 'through', 'a', 'low', 'lying', 'body', 'of', 'water'], 'raw': 'Two dogs running through a low lying body of water .', 'imgid': 0, 'sentid': 4}], 'split': 'train', 'filename': '2513260012_03d33305cf.jpg'}


In [70]:
from collections import Counter

train_image_paths = []
train_image_captions = []
test_image_paths = []
test_image_captions = []
word_freq = Counter()

max_len=50
for img in data['images']:
    captions = []
    for c in img['sentences']:
        word_freq.update(c['tokens'])
        if len(c['tokens']) <= max_len:
            captions.append(c['tokens'])
    if len(captions) == 0:
        continue
    path ="files/flickr8k_dataset/Images/"+img['filename']
    if img['split'] in {'train', 'val', 'restval'}:
        train_image_paths.append(path)
        train_image_captions.append(captions)
    elif img['split'] in {'test'}:
        test_image_paths.append(path)
        test_image_captions.append(captions)

In [71]:
assert len(train_image_paths)==len(train_image_captions)
assert len(test_image_paths)==len(test_image_captions)
print(f"there are {len(train_image_paths)} training images")
print(f"there are {len(test_image_paths)} test images")

there are 7000 training images
there are 1000 test images


In [72]:
min_word_freq = 0
words = [w for w in word_freq.keys() if word_freq[w]>min_word_freq]
word2idx = {k:v + 4 for v, k in enumerate(words)}
word2idx['<pad>'] = 0
word2idx['<start>'] = 1
word2idx['<end>'] = 2
word2idx['<unk>'] = 3

In [73]:
indexes = [word2idx.get(token, 3) for token in test_image_captions[0][0]]
print(indexes), print(test_image_captions[0][0]);

[12, 18, 318, 11, 12, 13, 11, 45, 30, 4, 234]
['the', 'dogs', 'are', 'in', 'the', 'snow', 'in', 'front', 'of', 'a', 'fence']


In [74]:
idx2word = {v:k for k, v in word2idx.items()}
print(f"there are {len(idx2word)} unique tokens")

there are 8387 unique tokens


In [75]:
tokens=[idx2word.get(idx,"<unk>") for
         idx in indexes]  #2
print(tokens)


['the', 'dogs', 'are', 'in', 'the', 'snow', 'in', 'front', 'of', 'a', 'fence']


In [76]:
!git clone https://github.com/markhliu/txt2img
import sys
sys.path.append("/content/txt2img")


fatal: destination path 'txt2img' already exists and is not an empty directory.


In [77]:
from txt2img.utils.caption_util import FlickrD  #1

trainset=FlickrD(train_image_paths,
  train_image_captions,word2idx)  #2
testset=FlickrD(test_image_paths,
  test_image_captions,word2idx)  #3

In [78]:
from torch.utils.data import DataLoader

BATCH_SIZE = 128
train_loader = DataLoader(trainset,
                          batch_size=BATCH_SIZE,
                          shuffle=True)  #1
test_loader = DataLoader(testset,
                        batch_size=BATCH_SIZE,
                        shuffle=True)  #2

# Transformer

In [79]:
import torch
import math
from torch import nn

In [80]:
def extract_patches(image_tensor, patch_size=8):
  bs, c, h, w = image_tensor.size()
  unfold = torch.nn.Unfold(kernel_size = patch_size, stride = patch_size)
  unfolded = unfold(image_tensor)
  unfolded = unfolded.transpose(1, 2)#.reshape(bs, -1, c* patch_size * patch_size)
  return unfolded

In [81]:
test_images,test_tokens,test_targets,test_mask=next(iter(test_loader))
image=test_images[0].unsqueeze(0)  #1
patches=extract_patches(image,patch_size=8)  #2
print(patches.shape)  #3

torch.Size([1, 256, 192])


In [82]:
class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, x):
        device = x.device
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = x[:, None] * emb[None, :]
        emb = torch.cat((emb.sin(), emb.cos()), dim=-1)
        return emb

In [83]:
class AttentionBlock(nn.Module):
  def __init__(self, hidden_size=128, num_heads=4, masking=True):
    super().__init__()
    self.masking = masking
    self.multihead_attn = nn.MultiheadAttention(hidden_size, num_heads, dropout=0.0, batch_first=True)

  def forward(self, x_in, kv_in, key_mask=None):
    if self.masking:
      bs, l, h = x_in.shape
      # The attn_mask should be (sequence_length, sequence_length)
      mask = torch.triu(torch.ones(l, l, device=x_in.device), 1).bool()
    else:
      mask = None
    return self.multihead_attn(x_in, kv_in, kv_in, attn_mask=mask, key_padding_mask=key_mask)[0]

In [93]:
class TransformerBlock(nn.Module):
  def __init__(self, hidden_size=128, num_heads=4, decoder=False, masking=True):
    super().__init__()
    self.decoder = decoder
    self.norm1 = nn.LayerNorm(hidden_size)
    self.attn1 = AttentionBlock(hidden_size, num_heads, masking) # Self-attention for decoder has masking

    if self.decoder:
      self.norm2 = nn.LayerNorm(hidden_size)
      self.attn2 = AttentionBlock(hidden_size, num_heads, masking=False) # Cross-attention does not need causal masking

    self.norm_mlp = nn.LayerNorm(hidden_size)
    self.mlp = nn.Sequential(nn.Linear(hidden_size, hidden_size * 4),
                             nn.ELU(),
                             nn.Linear(hidden_size * 4, hidden_size))

  def forward(self, x, input_key_mask=None, cross_key_mask=None, kv_cross=None):
    x = self.attn1(x, x, key_mask=input_key_mask) + x
    x = self.norm1(x)
    if self.decoder:
      x = self.attn2(x, kv_cross, key_mask=cross_key_mask) + x
      x = self.norm2(x)
    x = self.mlp(x)
    return x

In [101]:
class VisionEncoder(nn.Module):
  def __init__(self, image_size, channels_in, patch_size=16, hidden_size=128,
               num_layers=6, num_heads=4):
    super().__init__()
    self.patch_size = patch_size
    self.fc_in = nn.Linear(channels_in * patch_size * patch_size, hidden_size)
    seq_length = (image_size // patch_size) ** 2
    self.pos_embedding = nn.Parameter(torch.empty(1, seq_length, hidden_size).normal_(std=0.02))
    self.blocks = nn.ModuleList([TransformerBlock(hidden_size, num_heads, decoder=False, masking=False)
                          for _ in range(num_layers)])


  def forward(self, image):
    bs = image.shape[0]
    patch_seq = extract_patches(image, patch_size=self.patch_size)
    patch_emb = self.fc_in(patch_seq)
    embs = patch_emb + self.pos_embedding
    for block in self.blocks:
      embs = block(embs)
    return embs

In [102]:
class Decoder(nn.Module):
  def __init__(self, num_emb, hidden_size=128, num_layers=6, num_heads=4):
    super().__init__()
    self.embedding = nn.Embedding(num_emb, hidden_size)
    self.embedding.weight.data = 0.0001 * self.embedding.weight.data
    self.pos_emb = SinusoidalPosEmb(hidden_size)
    self.blocks = nn.ModuleList([TransformerBlock(hidden_size, num_heads, decoder=True)
                      for _ in range(num_layers)])
    self.fc_out = nn.Linear(hidden_size, num_emb)

  def forward(self, input_seq, encoder_output,
              input_padding_mask=None, encoder_padding_mask=None):
    input_embs = self.embedding(input_seq)
    bs, l, h = input_embs.shape
    seq_indx = torch.arange(l, device=input_seq.device)
    pos_emb = self.pos_emb(seq_indx).reshape(1, l, h).expand(bs, l, h)
    embs = input_embs + pos_emb
    for block in self.blocks:
      embs = block(embs,
                   input_key_mask = input_padding_mask,
                   cross_key_mask = encoder_padding_mask,
                   kv_cross = encoder_output)
    return self.fc_out(embs)

In [103]:
class VisionEncoderDecoder(nn.Module):
  def __init__(self, image_size, channels_in, num_emb,
               patch_size=16, hidden_size=128, num_layers=(6,6), num_heads=4):
    super().__init__()
    self.encoder = VisionEncoder(image_size, channels_in, patch_size, hidden_size, num_layers[0],  num_heads)
    self.decoder = Decoder(num_emb, hidden_size, num_layers[1], num_heads)

  def forward(self, input_image, target_seq, padding_mask):
    bool_padding_mask = padding_mask == 0
    encoded_seq = self.encoder(input_image)
    decoded_seq = self.decoder(target_seq, encoded_seq,
                               input_padding_mask=bool_padding_mask)
    return decoded_seq


# Training

In [104]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
hidden_size = 192
num_layers = (6,6)
num_heads = 8
patch_size = 8

print(device)

cuda


In [105]:
import torch
import gc

def free_gpu_memory():
    # Explicitly delete model and related tensors if they exist
    # Use globals().pop() to safely remove global variables and their references
    variables_to_delete = ['caption_model', 'images', 'inputs', 'outputs', 'masks', 'pred', 'loss']
    for var_name in variables_to_delete:
        if var_name in globals():
            del globals()[var_name]
            print(f"{var_name} deleted.")

    # Then clear CUDA cache and collect garbage
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("CUDA cache emptied.")
    gc.collect()
    print("Python garbage collected.")

free_gpu_memory()

caption_model deleted.
CUDA cache emptied.
Python garbage collected.


In [106]:
caption_model = VisionEncoderDecoder(
    image_size=128, channels_in=3,
    num_emb=len(word2idx), patch_size=patch_size,
    num_layers=num_layers, hidden_size=hidden_size,
    num_heads=num_heads
).to(device)

optimizer = torch.optim.Adam(caption_model.parameters(), lr=0.0001)
scaler = torch.amp.GradScaler('cuda')
loss_fn = nn.CrossEntropyLoss(reduction='none')

In [107]:
num_model_params = 0
for param in caption_model.parameters():
    num_model_params += param.flatten().shape[0]
print(f"This model has {num_model_params} parameters")

This model has 9545219 parameters


In [ ]:
from tqdm import tqdm

for epoch in range(0, 50):
  caption_model.train()
  eloss = 0
  for images, inputs, outputs, masks in tqdm(train_loader):
    images = images.to(device)
    tokens_in = inputs.to(device)
    padding_mask = masks.to(device)
    target_ids = outputs.to(device)
    with torch.amp.autocast('cuda'):
      pred = caption_model(images, tokens_in, padding_mask=padding_mask)
      loss = (loss_fn(pred.transpose(1,2), target_ids) * padding_mask).mean()

    optimizer.zero_grad()
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    eloss += loss.item()


  print(f'epoch {epoch}, loss is {eloss/len(train_loader)}')
torch.save(caption_model.state_dict(), 'files/caption.pth')

100%|██████████| 274/274 [04:28<00:00,  1.02it/s]


epoch 0, loss is 1.2735815618159998


 87%|████████▋ | 239/274 [04:00<00:37,  1.07s/it]

# Generation

In [ ]:
def caption(image,temp=1.0):
    # Add the Start-Of-Sentence token to the prompt
    sos_token = 1 * torch.ones(1, 1).long()
    log_tokens = [sos_token]
    caption_model.eval()
    with torch.no_grad():
        # Encode the input image
        image_embedding = caption_model.encoder(image.to(device))
        # Generate the caption tokens
        for i in range(50):
            input_tokens = torch.cat(log_tokens, 1)
            # Decode input tokens into the next predicted tokens
            data_pred = caption_model.decoder(
                input_tokens.to(device),image_embedding)
            # Sample from the distribution based on temperature
            dist = Categorical(logits=data_pred[:, -1] / temp)
            next_tokens = dist.sample().reshape(1, 1)
            # Append the next predicted token to the sequence
            log_tokens.append(next_tokens.cpu())
            # Stop if the End-Of-Caption token is predicted
            if next_tokens.item() == 2:
                break
    # Convert the list of token indices to a tensor
    pred_text = torch.cat(log_tokens, 1)
    pred_text_strings = [idx2word.get(i,"<unk>") for
                 i in pred_text[0].tolist() if i>3]
    # Join the token strings to form the predicted text
    pred_text = " ".join(pred_text_strings)
    return pred_text

In [ ]:
def compare(images, captions, index, temp=1.0):
    image = images[index].unsqueeze(0)  #1
    capi=captions[index]  #2
    capt=[idx2word.get(i,"UNK") for i in capi.tolist() if i>3]
    cap=" ".join(capt)
    pred=caption(image,temp=temp)  #3
    out=torchvision.utils.make_grid(image, 1, normalize=True)
    plt.figure(figsize=(5,10),dpi=100)
    out = torchvision.utils.make_grid(image, 1, normalize=True)
    plt.imshow(out.numpy().transpose((1, 2, 0)))
    plt.title(  #4
    f"**Original caption:\n"+cap+"\n**Generated caption:\n"+pred,
              wrap=True, loc="left", fontsize=18)
    plt.axis("off")
    plt.show()

In [ ]:
from torch.distributions import Categorical
import torchvision
import matplotlib.pyplot as plt

caption_model.load_state_dict(torch.load("files/caption.pth",
    weights_only=True,
    map_location=device))
compare(test_images, test_tokens, 0, temp=0.75)

In [ ]:
compare(test_images, test_tokens, 10, temp=0.75)

In [ ]:
compare(test_images, test_tokens, 20, temp=0.75)